# Group 3 | Seasonal Climatology

**Predicting Drought Before the Rains Fail**

District-level severe meteorological drought, Tanzania (1981–2025).

## 7. Results, interpretation and conclusion

**Study question:** How much advance drought-risk information can simple unweighted Seasonal Climatology provide at district level in Tanzania?

The baseline uses historical severe SPI-3 frequencies by district and calendar month. This notebook consolidates model construction, historical reconstruction and retrospective verification without adding any other forecast model.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
OUT = ROOT / "outputs"
for name in ["figures", "maps", "metrics", "predictions", "tables"]:
    (OUT / name).mkdir(parents=True, exist_ok=True)

import json
from IPython.display import display
import pandas as pd
from group3_climatology.pipeline import run_pipeline
from group3_climatology.config import INPUT, OUTPUT

summary=run_pipeline(INPUT,OUTPUT,plots=False)
qc=summary["data"]
history=summary["historical"]
case=summary["case_study_2020_2022"]
onset=summary["onset_2020_2022"]
print("Full reproducible project pipeline completed successfully.")
display(pd.DataFrame({"Indicator":["District-month records","Districts","Valid SPI-3","Severe SPI-3 months","Severe drought episodes","Longest severe episode (months)"],
    "Result":[qc["district_month_records"],qc["districts"],qc["valid_spi3"],history["severe_district_months"],history["severe_episodes"],history["longest_severe_episode_months"]]}))

Full reproducible project pipeline completed successfully.


,Indicator,Result
0,District-month records,81000
1,Districts,150
2,Valid SPI-3,80700
3,Severe SPI-3 months,4364
4,Severe drought episodes,2859
5,Longest severe episode (months),7


### Baseline forecast performance, 2020–2022

A warning threshold of 5.5% was selected from 2015–2018 validation targets. The 2020–2022 experiment contains 5,400 unique district-month observations at each lead.

In [2]:
main=pd.DataFrame({"Metric":["Observed severe district-months","Correct warnings","Missed drought observations","False warnings","Correct non-warnings","Recall","Precision","ROC-AUC","Brier Score"],
                   "Value":[case["severe_district_months"],case["true_alerts"],case["missed_drought_months"],case["false_alerts"],case["correct_nonalerts"],f"{case['recall']:.1%}",f"{case['precision']:.1%}",f"{case['roc_auc']:.3f}",f"{case['brier_score']:.4f}"]})
display(main)
print("Onset counts:",onset)

,Metric,Value
0,Observed severe district-months,150
1,Correct warnings,84
2,Missed drought observations,66
3,False warnings,2307
4,Correct non-warnings,2943
5,Recall,56.0%
6,Precision,3.5%
7,ROC-AUC,0.550
8,Brier Score,0.0287


Onset counts: {'observed_new_severe_onsets': 109, 'onset_months_with_warning': 55, 'onset_months_without_warning': 54}


### Findings

1. Seasonal Climatology provides reproducible district- and calendar-month-specific drought-risk probabilities without weights, smoothing or other model parameters.
2. Historical prevalence is not a reliable detector of rapidly changing climate conditions. The 2020–2022 ROC-AUC was approximately 0.55.
3. The validation-selected threshold identified 84 of 150 severe district-months, but issued 2,307 false warnings in the same period.
4. Forecast probabilities remain unchanged across the three lead times for a fixed district and target month. This represents an advance seasonal probability, not verified predictive awareness of a new drought onset.
5. The source SPI-3 classification does not identify severe months in Longido, Monduli or Simanjiro during 2020–2022, although seasonal rainfall deficits occurred. SPI-3 and independent impact evidence must be distinguished.

### Conclusion

Seasonal Climatology provides a transparent quantitative reference for district-level drought-risk forecasting. Its retrospective discriminatory power and alert precision are weak. The baseline is suitable for evaluating whether subsequent forecasting methods add value, but the evidence does not establish it as an operational early-warning service.

In [3]:
report=f"""# Group 3 — Seasonal Climatology results

## Data

- District-month records: {qc['district_month_records']:,}
- Districts: {qc['districts']}
- Valid SPI-3 values: {qc['valid_spi3']:,}
- Severe district-months (1981–2025): {history['severe_district_months']:,}
- Consecutive severe episodes: {history['severe_episodes']:,}

## Model

Unweighted seasonal frequency: p(d,m) = D(d,m)/N(d,m). Severe drought: SPI-3 <= -1.5.

## 2020–2022 retrospective verification (one lead)

- Observations: {case['forecasts']:,}
- Severe SPI-3 months: {case['severe_district_months']}
- Threshold selected on 2015–2018 validation: {summary['validation_selected_warning_threshold']:.1%}
- True alerts: {case['true_alerts']}
- Missed severe months: {case['missed_drought_months']}
- False alerts: {case['false_alerts']:,}
- Correct non-alerts: {case['correct_nonalerts']:,}
- Recall: {case['recall']:.2%}
- Precision: {case['precision']:.2%}
- ROC-AUC: {case['roc_auc']:.4f}
- Brier Score: {case['brier_score']:.5f}
- Onset months warned: {onset['onset_months_with_warning']} of {onset['observed_new_severe_onsets']}

## Limitations

This is a retrospective SPI-3 meteorological-drought evaluation. Seasonal probabilities do not adapt to current climate observations; 1–3-month lead equivalence is not a demonstrated early-detection lead time. Reference SPI-3 uses 1981–2010 climatological fits; the dataset applies 2022 district boundaries to earlier dates. No independently dated agricultural damage labels were used. The period 2020–2025 has previously been inspected and is not an untouched independent holdout.
"""
(ROOT / "docs" / "RESULTS.md").write_text(report,encoding="utf-8")
print("Reproducible results summary written to docs/RESULTS.md")

Reproducible results summary written to docs/RESULTS.md
